In [1]:
# Initialisation
#  echo: false
# Recharge automatiquement les modules modifiés (évite de redémarrer le kernel)
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.signal as sci
from scipy.signal import hilbert
from scipy.stats import kurtosis
from scipy.fftpack import fft, fftfreq
from IPython.display import Markdown

# Cherche le dossier racine du projet (BDS_Fault_Detection) pour faire les imports depuis Sources/
dossier = Path().resolve()
for parent in (dossier, *dossier.parents):
    if "BDS_Fault_Detection" in parent.name:
        RACINE = parent
sys.path.insert(0, str(RACINE))

# Importation des fonctions du projet (dossier Sources/). Attention de l'ordre d'importation  !
from Sources import data, config, signaux, indicateurs, features, models


I0000 00:00:1794838996.642587   97977 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
# Sélection du scope de données à traiter (sources, défauts, diamètres, positions, charges, canaux)
selection = data.selectionner(
    sources=("12kDE", "Normal"),
    defauts=("IR", "B", "OR"),
    diametres=(7, 14, 21), positions_or=("6",),
    charges=(0, 1, 2, 3),
    canaux={"12kDE": ("DE",), 
            "Normal": ("DE")},
)

In [3]:
from Sources.config import DUREE_FENETRE
from Sources.features import indicateurs_temporels, indicateurs_spectraux, indicateurs_spectraux_Mourad, creer_filtre, M_up_enveloppe_bande

def construire_table(selection, charger, canaux=None):
    """Étape 1 : une ligne par (fichier, canal), colonne 'signal' = signal BRUT entier + infos."""
    lignes = []
    for nom, info in selection.iterrows():
        liste = (info["canaux"] if "canaux" in selection.columns else None) if canaux is None else canaux
        if liste is None: liste = ("DE",)
        if isinstance(liste, str): liste = (liste,)
        s = charger(nom, canaux=liste)
        for canal in liste:
            lignes.append({"fichier": nom, "canal": canal, "fs": s["fs"], "rpm": s["rpm"],
                           "roulement": "FE" if "FE" in nom else "DE", "signal": s[canal],
                            "classe": "Sain" if info["defaut"] == "Sain" else info["defaut"] + str(info["diametre_mils"]).zfill(3),
                           **{k: v for k, v in info.to_dict().items() if k != "canaux"}})
    return pd.DataFrame(lignes)


def fenetrer_table(table, duree=DUREE_FENETRE, recouvrement=0.0):
    """Étape 2 : découpe chaque 'signal' en fenêtres → une ligne par (fichier, canal, fenêtre)."""
    lignes = []
    for _, r in table.iterrows():
        for i, w in enumerate(features.fenetrer(r["signal"], r["fs"], duree=duree, recouvrement=recouvrement)):
            lignes.append({**r.to_dict(), "fenetre": i, "signal": w})   # 'signal' = la fenêtre maintenant
    return pd.DataFrame(lignes)


def ajouter_indicateurs_temporels(table):
    inds = pd.DataFrame([indicateurs_temporels(w) for w in table["signal"]], index=table.index)
    return pd.concat([table, inds], axis=1)


def ajouter_indicateurs_spectraux(table):
    sos_cache, inds = {}, []
    for _, r in table.iterrows():
        if r["fs"] not in sos_cache: sos_cache[r["fs"]] = creer_filtre(r["fs"])
        inds.append(indicateurs_spectraux(r["signal"], r["fs"], r["rpm"], roulement=r["roulement"], sos=sos_cache[r["fs"]]))
    return pd.concat([table, pd.DataFrame(inds, index=table.index)], axis=1)


def ajouter_indicateurs_spectraux_Mourad(table):
    """Indicateurs Mourad sur chaque fenêtre (x_filt + enveloppe calculés par fenêtre)."""
    sos_cache, inds = {}, []
    for _, r in table.iterrows():
        if r["fs"] not in sos_cache: sos_cache[r["fs"]] = creer_filtre(r["fs"])
        x_filt, env = M_up_enveloppe_bande(r["signal"], r["fs"], sos=sos_cache[r["fs"]])
        inds.append(indicateurs_spectraux_Mourad(x_filt, env, r["fs"], r["rpm"], roulement=r["roulement"]))
    return pd.concat([table, pd.DataFrame(inds, index=table.index)], axis=1)

In [4]:
table = construire_table(selection, data.charger)
table = fenetrer_table(table, duree=DUREE_FENETRE)
table = ajouter_indicateurs_temporels(table)
table = ajouter_indicateurs_spectraux(table)
table = ajouter_indicateurs_spectraux_Mourad(table)

In [9]:
table.info()

<class 'pandas.DataFrame'>
RangeIndex: 2511 entries, 0 to 2510
Data columns (total 39 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   fichier        2511 non-null   str    
 1   canal          2511 non-null   str    
 2   fs             2511 non-null   int64  
 3   rpm            2511 non-null   float64
 4   roulement      2511 non-null   str    
 5   signal         2511 non-null   object 
 6   numero         2511 non-null   int64  
 7   source         2511 non-null   str    
 8   defaut         2511 non-null   str    
 9   position_OR    603 non-null    str    
 10  diametre_mils  2511 non-null   int64  
 11  charge_hp      2511 non-null   int64  
 12  rpm_nominal    2511 non-null   int64  
 13  fenetre        2511 non-null   int64  
 14  RMS            2511 non-null   float64
 15  Pic            2511 non-null   float64
 16  Crete          2511 non-null   float64
 17  Kurtosis       2511 non-null   float64
 18  amp_BPFI_1     2511